# DPWMS — Plastic Waste Hotspot Analysis

Runs the floating-debris pipeline (`plasticwaste_tool`) end to end:
Sentinel-2 fetch -> spectral indices -> water mask -> mean+3σ rule on FDI and NDVI
-> MARIDA debris classifier -> hotspot vectorisation -> stats -> validation -> monthly archive
-> national run and deployment preparation.

This is a SEPARATE module/screen from `run_analysis.ipynb` (Land Use & Deforestation), per the GEDH SRS
(Module 5 "Plastic Waste Section" vs Module 4 "Land Use and Deforestation Section"). It reuses the same
shared infrastructure (GEE service account, Azure Blob cache) that `landuse_tool` already established.

**Scientific basis**
- Floating Debris Index: Biermann, Clewley, Martinez-Vicente & Topouzelis (2020), *Finding Plastic
  Patches in Coastal Waters using Optical Satellite Data*, Sci. Rep. 10, 5364.
- Detection rule: Digital Earth Africa, *Detection of Floating Marine Debris* — FDI and NDVI both
  above mean + 3σ of the scene's water pixels.
- Debris classifier: a random forest trained on MARIDA (Kikaki et al. 2022, PLoS ONE 17(1):
  e0262247), built and measured in `marida_workflow.ipynb`.

> **The detector this notebook runs.** The FDI follows Biermann et al. (2020, eq. 1–2):
> NIR = **B8**, RE2 = **B6**, SWIR1 = **B11**, with the **B4 (red)** wavelength in the
> interpolation ratio (pinned by `plasticwaste_tool/test_fdi_formula.py`). A pixel is published
> when FDI and NDVI both exceed mean + 3σ of the AOI's water pixels **and** the MARIDA debris
> classifier gives it P(debris) ≥ 0.5 (`fdi_tool/debris_filter.py`).
>
> Figures produced before October 2026 used B8A in place of B6 and published the rule's
> detections without the classifier, so they are not comparable with the figures here. The
> formula id is part of every cache key, so composites from that version are never reused.

In [1]:
import sys, math
from pathlib import Path
from types import SimpleNamespace

ROOT = Path.cwd()                          # the folder holding plasticwaste_tool/
SCRIPTS_DIR = ROOT / "plasticwaste_tool"   # used by the review cells below
sys.path.insert(0, str(ROOT))

# Was: SCRIPTS_DIR = E:\SDG AI Lab\...\plasticwaste_tool and 00_environment_check.py,
# which exist only on the external drive (never committed). geo_env.fix_proj is
# the same PROJ_LIB repair and ships with plasticwaste_tool.
from plasticwaste_tool.geo_env import fix_proj
fix_proj(verbose=True)


def assert_pixel_area_sane(profile, scale_m_hint, tol=0.25):
    """QA only: the composite's pixels are about `scale_m_hint` metres across.

    A geographic (degree) grid is converted to metres at the raster's own centre
    latitude. Catches an export at the wrong scale, or a profile whose transform
    is in the wrong units, before any area is computed from it.
    """
    t, h, w = profile["transform"], profile["height"], profile["width"]
    if profile["crs"] and profile["crs"].is_geographic:
        lat = t.f + t.e * h / 2.0
        px_w = abs(t.a) * 111_320.0 * math.cos(math.radians(lat))
        px_h = abs(t.e) * 110_574.0
    else:
        px_w, px_h = abs(t.a), abs(t.e)
    for name, v in (("width", px_w), ("height", px_h)):
        if not (1 - tol) * scale_m_hint <= v <= (1 + tol) * scale_m_hint:
            raise AssertionError(f"pixel {name} {v:.1f} m, expected ~{scale_m_hint} m")
    print(f"pixel size OK: {px_w:.1f} x {px_h:.1f} m (expected ~{scale_m_hint} m)")


diag = SimpleNamespace(assert_pixel_area_sane=assert_pixel_area_sane)

PROJ_LIB: C:\Program Files\PostgreSQL\13\share\contrib\postgis-3.2\proj -> c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\rasterio\proj_data


In [2]:
import sys, os, json, logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

import numpy as np
import pandas as pd
import geopandas as gpd

from plasticwaste_tool import config as pwcfg
from plasticwaste_tool import api_connector as api
from plasticwaste_tool import data_loader as DL
from plasticwaste_tool import pipeline
from plasticwaste_tool import stats as ST
from plasticwaste_tool import validation as V

2026-10-02 17:28:25,363 INFO plasticwaste_tool.config: plasticwaste_tool: loaded shared .env from c:\Users\ASUS\Guinea_Project\Land Use-Deforestation\.env
2026-10-02 17:28:25,457 INFO landuse_tool.azure_utils: Loaded local .env from c:\Users\ASUS\Guinea_Project\Land Use-Deforestation\.env
2026-10-02 17:28:25,459 INFO landuse_tool.azure_utils: Loaded GEE service-account key from file.
2026-10-02 17:28:25,459 INFO landuse_tool.azure_utils: No AZURE_KEY_VAULT_URL set; using environment (and .env) as-is.
2026-10-02 17:28:26,683 INFO landuse_tool.azure_utils: Loaded local .env from c:\Users\ASUS\Guinea_Project\Land Use-Deforestation\.env
2026-10-02 17:28:26,683 INFO landuse_tool.azure_utils: No AZURE_KEY_VAULT_URL set; using environment (and .env) as-is.


## 1. Configuration — AOI, date window, credentials check

In [3]:
AOI_KEY = "conakry"                 # 'full' | 'conakry' | 'konkoure' | 'fatala' | 'mangrove'
START_DATE = "2024-01-01"
END_DATE = "2024-02-01"   # exclusive upper bound (GEE filterDate convention): the whole of
                          # January, the same "{next_month}-01" convention build_monthly_archive() uses.

problems = pwcfg.validate_required_keys()
if problems:
    print("CONFIGURATION PROBLEMS (fix before running against live GEE):")
    for p in problems:
        print(" -", p)
else:
    print("Credentials OK.")

pwcfg.ensure_directories()

Credentials OK.


## 1b. GEE service-account credentials — identical to run_analysis.ipynb's cell 8

Reads the `.json` key file **fresh, at runtime** (not a hand-pasted value in `.env`) so `GEE_SA_KEY`
is guaranteed to be valid JSON. This is the same trick `run_analysis.ipynb` uses; it's required because
`earthengine-api` only parses `GEE_SA_KEY` as JSON if `json.loads()` succeeds on it -- any hand-edited or
line-wrapped copy of the key in `.env` risks breaking the embedded PEM newlines/quotes and silently
falling through to a raw-PEM parser that will reject it.

In [4]:
from dotenv import load_dotenv

# 1. Explicitly point to the shared .env (same file landuse_tool/deforestation_tool use)
env_path = r"C:\Users\ASUS\Guinea_Project\Land Use-Deforestation\.env"
load_dotenv(dotenv_path=env_path, override=True)

# 2. Read the actual service-account JSON key file fresh -- do NOT rely on a
#    hand-pasted GEE_SA_KEY value sitting in .env; that's what breaks the PEM parsing.
full_key_path = r"C:\Users\ASUS\Guinea_Project\Land Use-Deforestation\landuse_tool\credentials\tribal-mapper-275801-7b57ce84f3bd.json"

try:
    with open(full_key_path, "r") as f:
        os.environ["GEE_SA_KEY"] = f.read()
    print("\u2705 Successfully loaded GEE key contents.")
except FileNotFoundError:
    print(f"\u274c Could not find the JSON key file at: {full_key_path}")

from landuse_tool.azure_utils import load_secrets_into_env
load_secrets_into_env()

problems = pwcfg.validate_required_keys()
if problems:
    print("Configuration problems:")
    for p in problems:
        print("  -", p.split(".")[0])
else:
    print("All required credentials present.")

2026-10-02 17:28:26,729 INFO landuse_tool.azure_utils: Loaded local .env from c:\Users\ASUS\Guinea_Project\Land Use-Deforestation\.env
2026-10-02 17:28:26,729 INFO landuse_tool.azure_utils: No AZURE_KEY_VAULT_URL set; using environment (and .env) as-is.


✅ Successfully loaded GEE key contents.
All required credentials present.


## 2. Initialize GEE + fetch the Sentinel-2 composite for the window

In [5]:
api.initialize_gee()
aoi = api.get_aoi_geometry(AOI_KEY)

gee_results = api.build_fdi_pipeline(START_DATE, END_DATE, aoi)
# Same key as pipeline.run_from_gee: the token hashes the AOI box AND the FDI
# formula id, so a composite cached under the old (B8A) index is never reused.
run_prefix = f"{AOI_KEY}_{api.aoi_cache_token(AOI_KEY)}_{START_DATE}_{END_DATE}"

# Multi-band download (raw bands + FDI/NDVI/NDWI) so the pipeline recomputes
# indices locally (testable / diffable against the GEE-side computation).
# force_refresh reverted to False (RUN_REVIEW_2_AND_PLATFORM.md action item #3):
# the scl_clear_classes land-mask fix already propagated through a forced
# refresh run, so the cache is now correct and normal caching should resume.
# force_refresh back to False: the WATER-band refresh has propagated (the run
# that produced the de_africa 79-patch result could not have worked without it),
# so the cached 10-band composite is correct and the slow per-band re-download
# is no longer needed. If a stale file ever does come back, read_composite
# raises a band-count error naming this call -- it fails loudly, not silently.
composite_path = api.download_composite_bands(gee_results, aoi, run_prefix, force_refresh=False)
print("Composite downloaded to:", composite_path)

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
2026-10-02 17:28:29,135 INFO landuse_tool.api_connector: GEE initialized via service account (guinea-environment-digital-hub@tribal-mapper-275801.iam.gserviceaccount.com).


Composite downloaded to: D:/gee_cache\cache\rasters/conakry_a410b95f_2024-01-01_2024-02-01_composite.tif


## 3. Run the offline pipeline (spectral indices -> water mask -> mean+3σ (FDI ∧ NDVI) -> MARIDA classifier -> hotspot vectors -> stats -> validation)

In [6]:
# pip uninstall -y rasterio pyproj fiona geopandas
# pip install --no-cache-dir rasterio pyproj fiona geopandas

In [7]:
bands, profile = DL.read_composite(composite_path)

diag.assert_pixel_area_sane(
    profile, scale_m_hint=pwcfg.S2_CONFIG["download_scale_m"])

cloud_pct = gee_results["cloud_pct"].getInfo()

# FIELD-VERIFIED reference points only -- points located INDEPENDENTLY of this
# detector (a ground survey, a prior confirmed report). None exist yet, so this
# stays None.
#
# It previously pointed at reference_points.csv, the 31 photo-interpreted
# positives. Those are centroids of the detector's OWN patches, so recall comes
# out 1.0 by construction and "precision" divides by all detected patches rather
# than the reviewed ones. The run before this change reported recall 1.0 /
# precision 0.57 / f1 0.726 with NO caveat (the provisional note is suppressed at
# n >= 30) and wrote it into metrics.json. validation.py now refuses that input
# outright, and this is None so it is never offered.
#
# The photo-interpreted review IS reported -- section 11, and metrics.json's
# separate `photo_interpretation_validation` key.
REFERENCE_POINTS_CSV = None
reference_points = None
if REFERENCE_POINTS_CSV and os.path.exists(REFERENCE_POINTS_CSV):
    ref_df = pd.read_csv(REFERENCE_POINTS_CSV)
    reference_points = gpd.GeoDataFrame(
        ref_df, geometry=gpd.points_from_xy(ref_df.longitude, ref_df.latitude), crs="EPSG:4326")

results = pipeline.run_from_arrays(
    bands, profile, acquisition_date=START_DATE, cloud_pct=cloud_pct,
    reference_points=reference_points)

print("Threshold used:", results["threshold"], f"({results['threshold_method']})")
print("Summary:", results["summary"])
print("SAR fallback recommended:", results["sar_fallback_recommended"])

pixel size OK: 19.7 x 19.9 m (expected ~20 m)


2026-10-02 17:28:30,912 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 242 of 6,558 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:31,366 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 242 of 6,558 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:31,504 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 482 of 7,537 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:31,638 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 97 of 5,777 rule pixels kept (P(debris) >= 0.50)


Threshold used: 0.06803719606250525 (de_africa_3sigma)
Summary: {'n_patches': 26, 'total_area_ha': 9.04, 'mean_area_ha': 0.3477, 'max_area_ha': 2.04, 'mean_concentration_score': 0.683, 'by_confidence': {'HIGH': 13, 'MEDIUM': 12, 'LOW': 1}}
SAR fallback recommended: False


### What the run produced

The log lines from `debris_filter` show the two stages separately. Over Conakry's water in January
2024 the FDI-and-NDVI rule flags **6,558 pixels**; the classifier keeps **242** of them, which
vectorise into **26 patches, 9.04 ha** (13 HIGH, 12 MEDIUM, 1 LOW).

The classifier removes most of what the rule flags because, with the FDI computed as Biermann et
al. define it, the rule responds to anything that lifts near-infrared above the water baseline —
vegetated shoreline included. `marida_workflow.ipynb` §4 measures this directly: the rule's pixels
have a median NDVI of 0.47 and 83% lie within 20 m of land; the pixels the classifier keeps have a
median NDVI of 0.10 and 2.5% lie within 20 m of land.

The log repeats for the threshold-sensitivity runs (±10%), which pass through the same classifier.

## 4. QA — cross-check locally-recomputed FDI vs the GEE server-side FDI
Small floating-point differences are expected (reduction order); large ones flag a band-order/mismatch bug.

In [8]:
from plasticwaste_tool import spectral as SP
qa = SP.compare_with_gee(results["fdi"], bands["FDI"])
print("FDI QA (local vs GEE):", qa)
assert qa["within_tolerance"], "Local and GEE FDI computations diverge beyond tolerance — investigate before trusting downstream stats."

FDI QA (local vs GEE): {'n_compared': 5169355, 'max_abs_diff': 5.960464477539063e-08, 'mean_abs_diff': 1.1608666250495503e-09, 'within_tolerance': True}


## 5. Threshold sensitivity + method consistency (report these in the methodology doc)

In [9]:
print("Threshold sensitivity (+/-10% perturbation of BOTH thresholds):")
for k, v in results["threshold_sensitivity"].items():
    print(f"  {k}: {v}")
print()
print("INTERPRETATION: report the PATCH COUNT change as the headline, not the area")
print("change. For a hotspot map the count is what decisions are made on, and area can")
print("move while the same places stay flagged. A large swing in patch count means the")
print("detection is sensitive to where the threshold lands -- call it provisional for")
print("this date/AOI.")
print()
print("NOTE: the older threshold_sensitivity_analysis() perturbed the FDI threshold")
print("alone. Under the de_africa rule that is NOT the deployed detector -- the NDVI")
print("conjunction carries the selectivity -- and its population was 33x larger than the")
print("rule actually selects. Any +/-% figure from a run before this change describes a")
print("different detector and must not be quoted.")

Threshold sensitivity (+/-10% perturbation of BOTH thresholds):
  rule: de_africa
  base_threshold_fdi: 0.06803719606250525
  base_threshold_ndvi: 0.09523680061101913
  base_area_px: 226
  base_n_patches: 26
  perturbation_frac: 0.1
  lower_area_px: 457
  lower_n_patches: 32
  lower_area_change_pct: 102.2
  lower_patch_count_change_pct: 23.1
  upper_area_px: 86
  upper_n_patches: 14
  upper_area_change_pct: -61.9
  upper_patch_count_change_pct: -46.2

INTERPRETATION: report the PATCH COUNT change as the headline, not the area
change. For a hotspot map the count is what decisions are made on, and area can
move while the same places stay flagged. A large swing in patch count means the
detection is sensitive to where the threshold lands -- call it provisional for
this date/AOI.

NOTE: the older threshold_sensitivity_analysis() perturbed the FDI threshold
alone. Under the de_africa rule that is NOT the deployed detector -- the NDVI
conjunction carries the selectivity -- and its population 

## 6. Reference-point validation (if reference points were supplied above)

In [10]:
if "reference_validation" in results:
    print("Reference-point validation:", results["reference_validation"])
else:
    print("No reference points supplied for this run — precision/recall not computed.")
    print("NOTE FOR METHODOLOGY DOC: do not report an accuracy figure for hotspot")
    print("detection until at least a handful of field-verified points exist; the")
    print("anomaly threshold's own statistics are not a substitute for ground truth.")

No reference points supplied for this run — precision/recall not computed.
NOTE FOR METHODOLOGY DOC: do not report an accuracy figure for hotspot
detection until at least a handful of field-verified points exist; the
anomaly threshold's own statistics are not a substitute for ground truth.


## 7. Hotspot patch table + map preview

In [11]:
hotspots = results["hotspot_vectors"]
display(hotspots[["area_m2", "floating_debris_score", "confidence_level"]]
        .sort_values("area_m2", ascending=False).head(20) if not hotspots.empty else "No patches detected.")

,area_m2,floating_debris_score,confidence_level
8,20400.186631,0.749017,HIGH
3,10800.098805,0.669065,HIGH
2,10000.091486,0.885191,HIGH
4,10000.091486,0.645237,MEDIUM
10,6000.054892,0.800390,HIGH
9,3600.032935,0.332970,MEDIUM
1,3200.029275,1.000000,HIGH
0,2400.021957,0.898595,HIGH
21,2400.021957,0.879595,HIGH
16,2000.018297,0.579513,MEDIUM


In [12]:
# --- Verify the FDI-intensity concentration score (detection.compute_concentration_score) ---
# Confirms the fix in detection.py section 5b: floating_debris_score should now
# come from FDI intensity inside each patch, not a rescaling of area_m2, and
# confidence_level should spread across LOW/MEDIUM/HIGH instead of landing in one band.
print("Concentration score (FDI-intensity-based) stats:")
print(hotspots["floating_debris_score"].describe() if not hotspots.empty else "No patches detected.")

print("\nConfidence level distribution:")
print(hotspots["confidence_level"].value_counts() if not hotspots.empty
      else "No patches detected.")

if not hotspots.empty and hotspots["confidence_level"].nunique() <= 1:
    print(f"\nWARNING: all patches still land in a single confidence band "
          f"({hotspots['confidence_level'].unique()[0]!r}). The "
          f"'confidence_band_low_max'/'confidence_band_high_min' thresholds in "
          f"DETECTION_CONFIG need recalibrating to this scene's score range — "
          f"document the change, don't silently retune.")

Concentration score (FDI-intensity-based) stats:
count    26.000000
mean      0.682979
std       0.193992
min       0.318053
25%       0.526545
50%       0.659430
75%       0.864392
max       1.000000
Name: floating_debris_score, dtype: float64

Confidence level distribution:
confidence_level
HIGH      13
MEDIUM    12
LOW        1
Name: count, dtype: int64


## 8. Monthly FDI Archive (GEDH-FR-0507) — build a short back-run for the archive slider

In [13]:
ARCHIVE_MONTHS = [(2023, 11), (2023, 12), (2024, 1)]   # (year, month) tuples

# force_refresh is back to False and can stay there. build_monthly_archive now
# puts the detection rule in its cache key, so the de_africa (FDI, NDVI, WATER)
# stacks are fetched fresh on their own keys and the old single-band Otsu-era
# files are never returned by mistake -- no forced refresh needed to shake them
# out. Each month is one GEE fetch of its raw bands, FDI, NDVI and WATER, the
# first time only.
archive_paths = api.build_monthly_archive(ARCHIVE_MONTHS, aoi_key=AOI_KEY, force_refresh=False)

ARCHIVE_RULE = pwcfg.DETECTION_CONFIG.get("detection_rule", "de_africa")
monthly_hotspots, archive_thresholds = {}, {}
for label, path in archive_paths.items():
    if ARCHIVE_RULE == "de_africa":
        ab, prof = DL.read_composite(
            path, band_names=pwcfg.ARCHIVE_CONFIG["bands_de_africa"])
        # The same function the main composite (section 3) and the platform
        # run: FDI from the raw bands, the mean+3sigma FDI-and-NDVI rule over the
        # whole water population, then the MARIDA debris classifier. One path,
        # so the 2024-01 archive month must reproduce section 3 exactly.
        res = pipeline.run_from_arrays(ab, prof, label)
        thresh = res["threshold"]
        archive_thresholds[label] = {"fdi": thresh, "ndvi": res.get("threshold_ndvi")}
        monthly_hotspots[label] = res["hotspot_vectors"]
        method = res["threshold_method"]
        kept = res.get("debris_filter", {})
        note = (f"rule {kept['n_px_rule']:>6,} px -> kept {kept['n_px_kept']:>5,} px"
                if kept else "no debris filter")
    else:
        score_fdi, prof = DL.read_single_band(path)
        mask, thresh, method = pipeline.D.hotspot_binary_mask(score_fdi)
        archive_thresholds[label] = {"fdi": thresh}
        monthly_hotspots[label] = pipeline.D.vectorize_hotspots(
            mask, prof, label, masked_fdi=score_fdi, threshold=thresh)
        note = ""
    print(f"{label}: {method:<18} t_fdi={thresh:+.5f} "
          f"t_ndvi={archive_thresholds[label].get('ndvi', float('nan')):+.5f} "
          f"patches={len(monthly_hotspots[label]):>5}  {note}")

time_series = ST.monthly_time_series(monthly_hotspots)
display(time_series)
DL.update_archive_index(archive_paths)

2026-10-02 17:28:32,709 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 317 of 15,536 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:33,315 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 317 of 15,536 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:33,472 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 816 of 17,667 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:33,623 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 113 of 13,716 rule pixels kept (P(debris) >= 0.50)


2023-11: de_africa_3sigma   t_fdi=+0.06969 t_ndvi=+0.11487 patches=   44  rule 15,536 px -> kept   317 px


2026-10-02 17:28:34,573 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 0 of 9,329 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:34,770 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 0 of 9,329 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:34,904 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 8 of 11,210 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:35,054 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 0 of 8,273 rule pixels kept (P(debris) >= 0.50)


2023-12: de_africa_3sigma   t_fdi=+0.06981 t_ndvi=+0.14795 patches=    0  rule  9,329 px -> kept     0 px


2026-10-02 17:28:35,931 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 242 of 6,558 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:36,395 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 242 of 6,558 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:36,536 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 482 of 7,537 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:36,670 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 97 of 5,777 rule pixels kept (P(debris) >= 0.50)


2024-01: de_africa_3sigma   t_fdi=+0.06804 t_ndvi=+0.09524 patches=   26  rule  6,558 px -> kept   242 px


,year_month,n_patches,total_area_ha,mean_area_ha,max_area_ha,mean_concentration_score,by_confidence,month_over_month_area_change_pct
0,2023-11,44,10.56,0.2400,1.04,0.752,"{'HIGH': 31, 'MEDIUM': 13}",NaN
1,2023-12,0,0.00,NaN,NaN,NaN,NaN,-100.0
2,2024-01,26,9.04,0.3477,2.04,0.683,"{'HIGH': 13, 'MEDIUM': 12, 'LOW': 1}",inf


{'2023-11': 'D:/gee_cache\\cache\\archive/fdi/conakry_a410b95f_2023-11_de_africa_10b.tif',
 '2023-12': 'D:/gee_cache\\cache\\archive/fdi/conakry_a410b95f_2023-12_de_africa_10b.tif',
 '2024-01': 'D:/gee_cache\\cache\\archive/fdi/conakry_a410b95f_2024-01_de_africa_10b.tif'}

In [14]:
MAIN_2024_01 = {"n_patches": results["summary"]["n_patches"],
                "total_area_ha": results["summary"]["total_area_ha"]}
row = time_series[time_series.year_month == "2024-01"].iloc[0]

print(f"main run    : {MAIN_2024_01['n_patches']:>6} patches, "
      f"{MAIN_2024_01['total_area_ha']:>12,.1f} ha")
print(f"archive run : {row.n_patches:>6} patches, {row.total_area_ha:>12,.1f} ha")
print(f"ratio       : {row.total_area_ha/MAIN_2024_01['total_area_ha']:,.3f}x")

aoi_ha = 5_169_292 * 400 / 10_000
print(f"largest patch = {row.max_area_ha/aoi_ha:.1%} of the AOI")

# Both paths now run the SAME rule over the SAME pixel population: the main
# path recomputes FDI/NDVI locally from the raw composite bands, the archive
# reads GEE's own FDI/NDVI bands from the monthly stack. So this is a genuine
# cross-check of the server-side and offline index computations, not just a
# plumbing test -- a small residual is expected and informative, a large one is
# structural.
_rel = abs(row.total_area_ha - MAIN_2024_01["total_area_ha"]) / MAIN_2024_01["total_area_ha"]

assert row.total_area_ha < MAIN_2024_01["total_area_ha"] * 2, (
    f"Archive mapped {row.total_area_ha:,.1f} ha against the main path's "
    f"{MAIN_2024_01['total_area_ha']:,.1f} ha for the same AOI and month. A gap this "
    "size means the two paths are not running the same detection rule -- check that "
    "both report the same threshold_method, and that the kernel was restarted after "
    "the last module edit. Do not use the monthly trend figure until these agree.")

if _rel > 0.10:
    print(f"\nWARNING: {_rel:.1%} difference between the main and archive paths for "
          f"2024-01.\n  Both apply the same rule, so this is index-computation drift "
          f"(local recompute\n  vs GEE-side bands) amplified by the mean+3*sigma "
          f"threshold, not a wrong mask.\n  Report the main-path figures and note the "
          f"archive spread as a reproducibility caveat.")
else:
    print(f"\nmain vs archive: {_rel:.2%} difference -- the two paths agree.")

main run    :     26 patches,          9.0 ha
archive run :     26 patches,          9.0 ha
ratio       : 1.000x
largest patch = 0.0% of the AOI

main vs archive: 0.00% difference -- the two paths agree.


### Main run and archive: one window convention, one detector

Both paths build a month as `start = YYYY-MM-01`, `end = first day of the next month`. Earth
Engine's `filterDate` end bound is exclusive, so that is the whole calendar month, and `END_DATE`
in section 1 follows the same convention. Both paths then run `pipeline.run_from_arrays` — the
rule, then the classifier — on the same bands; the main path recomputes FDI and NDVI locally, the
archive reads Earth Engine's own. The cell above is therefore a cross-check of the two index
computations: January 2024 gives **26 patches / 9.0 ha on both paths**.

In [15]:
# --- Per-month outlier check (the Jan 2024 assert above only checks Jan vs. the
# main run — it says nothing about whether Nov/Dec are internally sane) ---
# Flags a month if one patch dominates its own total area: a single large patch
# can drive the month's total by itself, so it is inspected before the trend is used.
DOMINANCE_THRESHOLD = 0.10  # one patch >10% of that month's own mapped area is suspicious

print(f"{'month':<10} {'n_patches':>10} {'total_ha':>10} {'max_ha':>10} {'max/total':>10}")
for _, r in time_series.iterrows():
    frac = r.max_area_ha / r.total_area_ha if r.total_area_ha else 0.0
    flag = "  <-- CHECK (one patch dominates; inspect it before using the trend)" \
        if frac > DOMINANCE_THRESHOLD else ""
    print(f"{r.year_month:<10} {r.n_patches:>10} {r.total_area_ha:>10.1f} "
          f"{r.max_area_ha:>10.1f} {frac:>9.1%}{flag}")

print("\nIf a month is flagged, plot its largest patch against the true-color composite")
print("for that window before trusting the monthly trend figure — a single outlier patch")
print("can swing 'total_area_ha' by itself and make an otherwise-clean series look volatile.")

month       n_patches   total_ha     max_ha  max/total
2023-11            44       10.6        1.0      9.8%
2023-12             0        0.0        nan      0.0%
2024-01            26        9.0        2.0     22.6%  <-- CHECK (one patch dominates; inspect it before using the trend)

If a month is flagged, plot its largest patch against the true-color composite
for that window before trusting the monthly trend figure — a single outlier patch
can swing 'total_area_ha' by itself and make an otherwise-clean series look volatile.


### Largest-patch check: December 2023

Fetches a small Sentinel-2 true-colour chip around the month's largest published patch and
overlays its boundary. A genuine floating-debris feature should look like discrete bright streaks
or rafts on otherwise clear water; cloud, cloud shadow, sun glint, turbid plumes and the shoreline
each have recognisable signatures that are not debris.

December 2023 has **no published patches** under the current detector, so the cell reports that
and stops.

In [16]:
import ee
import requests
import rasterio
import numpy as np
import matplotlib.pyplot as plt

dec_hotspots = monthly_hotspots["2023-12"]
QA_DIR = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"], "figures")
os.makedirs(QA_DIR, exist_ok=True)
if dec_hotspots.empty:
    print("No published patches in 2023-12 -- nothing to inspect.")
else:
    outlier_patch = dec_hotspots.loc[dec_hotspots.area_m2.idxmax()]
    print(f"Largest December 2023 patch: {outlier_patch.area_m2 / ST.M2_PER_HA:.1f} ha "
         f"(concentration score {outlier_patch.floating_debris_score:.3f})")

    # Crop tightly around the patch, padded 2x its own width/height for visual
    # context (shoreline, surrounding water) -- NOT the whole AOI, keeps this fast.
    minx, miny, maxx, maxy = outlier_patch.geometry.bounds
    pad_x, pad_y = (maxx - minx) * 2, (maxy - miny) * 2
    region = ee.Geometry.Rectangle(
        [minx - pad_x, miny - pad_y, maxx + pad_x, maxy + pad_y])

    s2_dec = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterDate("2023-12-01", "2024-01-01")
             .filterBounds(region)
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",
                                  pwcfg.DETECTION_CONFIG["sar_fallback_cloud_pct"]))
             .median())
    rgb = s2_dec.select(["B4", "B3", "B2"]).toFloat()

    # Note: this cell runs before Section 10 defines FIG_DIR/plt/np, so it imports
    # and defines its own paths rather than depending on later cells.
    chip_path = os.path.join(pwcfg.PROCESSING_CONFIG["cache_dir"], "qa", "dec2023_outlier_rgb.tif")
    os.makedirs(os.path.dirname(chip_path), exist_ok=True)
    url = rgb.getDownloadURL({"scale": 10, "region": region, "format": "GEO_TIFF", "crs": "EPSG:4326"})
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    with open(chip_path, "wb") as fh:
        fh.write(resp.content)

    with rasterio.open(chip_path) as src:
        rgb_arr = src.read()  # (3, H, W), raw S2 SR digital numbers
        extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

    # Raw SR digital numbers (~0-10000 scale) look near-black displayed linearly;
    # stretch to a typical true-colour range (matches 05_guinea_inference.py's S2
    # vis params: min=0, max=3000) and gamma-correct for a natural look.
    disp = np.clip(rgb_arr / 3000.0, 0, 1) ** (1 / 1.8)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(np.moveaxis(disp, 0, -1), extent=extent)

    outlier_gdf = gpd.GeoDataFrame([outlier_patch], geometry="geometry", crs=dec_hotspots.crs)
    outlier_gdf.boundary.plot(ax=ax, color="red", linewidth=2)
    ax.set_title(f"December 2023 outlier patch ({outlier_patch.area_m2 / ST.M2_PER_HA:.1f} ha) "
                f"vs Sentinel-2 true colour")
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")

    fig.tight_layout()
    fig.savefig(os.path.join(QA_DIR, "dec2023_outlier_qa.png"), dpi=200, bbox_inches="tight")
    plt.show()

    print("\nVerdict: record in the methodology doc whether the outlined patch aligns with")
    print("a cloud/cloud-shadow streak, sun glint, a turbid sediment plume, or looks like a")
    print("genuine isolated floating-debris feature. Do not use the December trend figure")
    print("in the report until this is recorded.")

No published patches in 2023-12 -- nothing to inspect.


### Largest-patch check: January 2024

The dominance check above flags January: its largest patch is **2.0 ha, 22.6% of the month's
mapped area**. The same patch is `patch_HIGH_8` in the photo review (section 11), where it is
labelled *ambiguous* — near-shore water with land in view.

In [17]:
jan_hotspots = monthly_hotspots["2024-01"]
QA_DIR = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"], "figures")
os.makedirs(QA_DIR, exist_ok=True)
if jan_hotspots.empty:
    print("No published patches in 2024-01 -- nothing to inspect.")
else:
    jan_outlier = jan_hotspots.loc[jan_hotspots.area_m2.idxmax()]
    print(f"Largest January 2024 patch: {jan_outlier.area_m2 / ST.M2_PER_HA:.1f} ha "
         f"(concentration score {jan_outlier.floating_debris_score:.3f})")

    minx, miny, maxx, maxy = jan_outlier.geometry.bounds
    pad_x, pad_y = (maxx - minx) * 2, (maxy - miny) * 2
    region_jan = ee.Geometry.Rectangle(
        [minx - pad_x, miny - pad_y, maxx + pad_x, maxy + pad_y])

    s2_jan = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterDate("2024-01-01", "2024-02-01")
             .filterBounds(region_jan)
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",
                                  pwcfg.DETECTION_CONFIG["sar_fallback_cloud_pct"]))
             .median())
    rgb_jan = s2_jan.select(["B4", "B3", "B2"]).toFloat()

    chip_path_jan = os.path.join(pwcfg.PROCESSING_CONFIG["cache_dir"], "qa", "jan2024_outlier_rgb.tif")
    os.makedirs(os.path.dirname(chip_path_jan), exist_ok=True)
    url = rgb_jan.getDownloadURL({"scale": 10, "region": region_jan, "format": "GEO_TIFF", "crs": "EPSG:4326"})
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    with open(chip_path_jan, "wb") as fh:
        fh.write(resp.content)

    with rasterio.open(chip_path_jan) as src:
        rgb_arr_jan = src.read()
        extent_jan = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

    disp_jan = np.clip(rgb_arr_jan / 3000.0, 0, 1) ** (1 / 1.8)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(np.moveaxis(disp_jan, 0, -1), extent=extent_jan)

    jan_gdf = gpd.GeoDataFrame([jan_outlier], geometry="geometry", crs=jan_hotspots.crs)
    jan_gdf.boundary.plot(ax=ax, color="red", linewidth=2)
    ax.set_title(f"January 2024 outlier patch ({jan_outlier.area_m2 / ST.M2_PER_HA:.1f} ha) "
                f"vs Sentinel-2 true colour")
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")

    fig.tight_layout()
    fig.savefig(os.path.join(QA_DIR, "jan2024_outlier_qa.png"), dpi=200, bbox_inches="tight")
    plt.show()

    print("\nSame verdict question as December: cloud/shadow/glint/sediment, or genuine debris?")

Largest January 2024 patch: 2.0 ha (concentration score 0.749)

Same verdict question as December: cloud/shadow/glint/sediment, or genuine debris?


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22840\1574429069.py:49: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Largest-patch check for any month (reusable)

`inspect_outlier("YYYY-MM")` does the same job for any month: the largest published patch against
the Sentinel-2 true-colour composite for that window. It is run below for November 2023, the month
with the most published patches of the three.

**What you are deciding:** does the outlined patch align with a cloud or cloud-shadow streak, sun
glint, a turbid sediment plume, a coastal-land or built-structure edge, or does it read as isolated
floating material?

**What to record:** verdict (`positive` / `negative` / `ambiguous`), interpreter confidence, and
what the feature looks like, in `plasticwaste_tool/manual_qa/fdi/outlier_verdicts.md`.

In [18]:
def inspect_outlier(month_label, pad_factor=2.0, scale_m=10):
    """Largest patch of `month_label` (YYYY-MM) against Sentinel-2 true colour.

    Generalizes the two hardcoded cells above so any month the dominance check
    flags can be interpreted the same way, with the same evidence on screen.
    """
    if monthly_hotspots[month_label].empty:
        print(f"No published patches in {month_label} -- nothing to inspect.")
        return None
    gdf = monthly_hotspots[month_label]
    if gdf.empty:
        print(f"{month_label}: no patches to inspect.")
        return None

    patch = gdf.loc[gdf.area_m2.idxmax()]
    share = patch.area_m2 / gdf.area_m2.sum()
    print(f"Largest {month_label} patch: {patch.area_m2 / ST.M2_PER_HA:.2f} ha "
          f"({share:.1%} of that month's mapped area, "
          f"concentration score {patch.floating_debris_score:.3f})")

    y, m = int(month_label[:4]), int(month_label[5:7])
    start = f"{y:04d}-{m:02d}-01"
    end = f"{y + (1 if m == 12 else 0):04d}-{(m % 12) + 1:02d}-01"

    minx, miny, maxx, maxy = patch.geometry.bounds
    pad_x, pad_y = (maxx - minx) * pad_factor, (maxy - miny) * pad_factor
    region = ee.Geometry.Rectangle([minx - pad_x, miny - pad_y,
                                    maxx + pad_x, maxy + pad_y])

    rgb = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(start, end)          # same exclusive-end convention as the archive
           .filterBounds(region)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",
                                pwcfg.DETECTION_CONFIG["sar_fallback_cloud_pct"]))
           .median().select(["B4", "B3", "B2"]).toFloat())

    chip = os.path.join(pwcfg.PROCESSING_CONFIG["cache_dir"], "qa",
                        f"{month_label}_outlier_rgb.tif")
    os.makedirs(os.path.dirname(chip), exist_ok=True)
    url = rgb.getDownloadURL({"scale": scale_m, "region": region,
                              "format": "GEO_TIFF", "crs": "EPSG:4326"})
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    with open(chip, "wb") as fh:
        fh.write(resp.content)

    with rasterio.open(chip) as src:
        arr = src.read()
        extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

    # Raw SR digital numbers (~0-10000) render near-black linearly; stretch to
    # min=0/max=3000 and gamma-correct, matching the other true-colour chips.
    disp = np.clip(arr / 3000.0, 0, 1) ** (1 / 1.8)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(np.moveaxis(disp, 0, -1), extent=extent)
    gpd.GeoDataFrame([patch], geometry="geometry", crs=gdf.crs) \
        .boundary.plot(ax=ax, color="red", linewidth=2)
    ax.set_title(f"{month_label} outlier patch "
                 f"({patch.area_m2 / ST.M2_PER_HA:.2f} ha, {share:.1%} of month) "
                 f"vs Sentinel-2 true colour")
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
    fig.tight_layout()
    out = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"], "figures",
                       f"{month_label}_outlier_qa.png")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")
    return patch


# November 2023: the month with the most published patches of the three.
_ = inspect_outlier("2023-11")

print()
print("Record your verdict in plasticwaste_tool/manual_qa/fdi/outlier_verdicts.md:")
print("  month | largest patch ha | share | verdict | confidence | what it looks like")
print()
print("Re-run with any other month label to check it, e.g. inspect_outlier('2023-12').")

Largest 2023-11 patch: 1.04 ha (9.8% of that month's mapped area, concentration score 0.676)
Saved D:/gee_cache\processed\figures\2023-11_outlier_qa.png

Record your verdict in plasticwaste_tool/manual_qa/fdi/outlier_verdicts.md:
  month | largest patch ha | share | verdict | confidence | what it looks like

Re-run with any other month label to check it, e.g. inspect_outlier('2023-12').


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22840\2362904965.py:68: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Photo-interpreted review of the published patches (no field data yet)

Two samples, each saved as a true-colour chip (Sentinel-2 monthly median, 10 m) for review by eye:

1. **Every published patch** (full enumeration), grouped by confidence level — the precision side:
   does the outlined patch look like floating material, or like a confounder (shoreline, cloud or
   shadow, glint, sediment plume, structure, vessel)?
2. **Ten random unflagged locations inside the AOI** — the recall side: did the layer miss anything
   visibly debris-like?

The first cell below checks whether the labelled file on disk
(`plasticwaste_tool/manual_qa/fdi/reference_points_labelled.csv`) belongs to this run's patches —
same number of patches, and the labelled points fall on them. Only when it does not are chips and a
blank template generated (about 40 Earth Engine downloads). Label each row `positive`, `negative` or
`ambiguous`, with `interpreter_confidence`, `material_guess` and a short `evidence_note`, and save the
filled template under that name.

In [19]:
# Does the labelled review on disk belong to THIS run's patches?
#
# A review is a judgement about specific patches. When the detector changes --
# the FDI band correction and the MARIDA debris filter both did -- the patches
# change, and labels made for the old ones say nothing about the new ones.
# Quoting them would report the precision of a layer that is no longer
# published. So: the review counts only if it enumerates the current patches
# (same number) and its points fall on them. Otherwise the chip cell below
# generates a fresh review for the current patches.
LABELS_CSV = os.path.join(SCRIPTS_DIR, "manual_qa", "fdi", "reference_points_labelled.csv")


def labels_match_patches(csv_path, patches, tol_m=30.0, min_share=0.9):
    if not os.path.exists(csv_path):
        return False, "no labelled review on disk"
    if patches.empty:
        return False, "this run has no patches"
    lab = pd.read_csv(csv_path)
    lab = lab[lab.sample_type == "flagged_patch"]
    utm = patches.estimate_utm_crs()
    pts = gpd.GeoSeries(gpd.points_from_xy(lab.longitude, lab.latitude),
                        crs="EPSG:4326").to_crs(utm)
    on_patch = float(pts.within(patches.to_crs(utm).buffer(tol_m).union_all()).mean())
    ok = len(lab) == len(patches) and on_patch >= min_share
    return ok, (f"{len(lab)} labelled patches vs {len(patches)} in this run; "
                f"{on_patch:.0%} of labelled points fall on a current patch")


labels_ok, why = labels_match_patches(LABELS_CSV, hotspots)
REVIEW_NEEDED = not labels_ok
print(("Labelled review matches this run: " if labels_ok else
       "Labelled review is for a DIFFERENT patch set -- not used: ") + why)
if REVIEW_NEEDED:
    print("The next cell generates chips and a blank template for the current "
          f"{len(hotspots)} patches. Label them, then save the filled template as\n  {LABELS_CSV}")

Labelled review matches this run: 26 labelled patches vs 26 in this run; 100% of labelled points fall on a current patch


In [20]:
# Runs only when the review on disk does not match this run's patches
# (REVIEW_NEEDED, cell above): one chip per patch, full enumeration.
if REVIEW_NEEDED:
    import random

    # None = review every detected patch (full enumeration -- no sampling
    # uncertainty, and every patch contributes pixels to the NDVI-FDI training set).
    # Set an integer to fall back to a stratified sample per confidence band if a
    # future run produces more patches than is practical to review by eye.
    N_PER_BAND = None
    N_RECALL_CHECK = 10     # unflagged locations to spot-check for missed debris
    REF_DIR = os.path.join(pwcfg.PROCESSING_CONFIG["cache_dir"], "qa", "reference_candidates")
    os.makedirs(REF_DIR, exist_ok=True)
    rng = random.Random(42)


    def _fetch_chip(region, out_path, date_start, date_end):
        s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterDate(date_start, date_end)
              .filterBounds(region)
              .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",
                                   pwcfg.DETECTION_CONFIG["sar_fallback_cloud_pct"]))
              .median())
        rgb = s2.select(["B4", "B3", "B2"]).toFloat()
        url = rgb.getDownloadURL({"scale": 10, "region": region, "format": "GEO_TIFF", "crs": "EPSG:4326"})
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        with open(out_path, "wb") as fh:
            fh.write(resp.content)


    def _save_chip_png(tif_path, geom_gdf, png_path, title):
        with rasterio.open(tif_path) as src:
            arr = src.read()
            extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
        disp = np.clip(arr / 3000.0, 0, 1) ** (1 / 1.8)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(np.moveaxis(disp, 0, -1), extent=extent)
        if geom_gdf is not None:
            geom_gdf.boundary.plot(ax=ax, color="red", linewidth=2)
        ax.set_title(title, fontsize=9)
        ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
        fig.tight_layout()
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.close(fig)


    from rasterio.features import geometry_mask

    rows, pixel_rows, n_failed = [], [], 0
    _fdi_arr, _ndvi_arr = results["fdi"], results["ndvi"]
    _transform = profile["transform"]


    def _patch_spectra(geom):
        """Every pixel inside one patch, as (ndvi, fdi) pairs.

        This is the feature space Biermann et al. (2020) classify in: they separate
        plastic from seaweed, timber, foam and seawater with a naive Bayes model
        over 2-D NDVI-FDI, trained on 53/48/60/17/20 pixels per class. Collecting
        the pixels now costs nothing, since the chips are being generated anyway.
        """
        inside = geometry_mask([geom], out_shape=_fdi_arr.shape, transform=_transform,
                               invert=True)
        ok = inside & np.isfinite(_fdi_arr) & np.isfinite(_ndvi_arr)
        return _ndvi_arr[ok], _fdi_arr[ok]

    # --- 1. Stratified sample of flagged patches (precision-side) ---
    for band in ["HIGH", "MEDIUM", "LOW"]:
        band_patches = hotspots[hotspots.confidence_level == band]
        if band_patches.empty:
            continue
        sample = (band_patches if N_PER_BAND is None
                  else band_patches.sample(n=min(N_PER_BAND, len(band_patches)),
                                           random_state=42))
        for idx, patch in sample.iterrows():
            pid = f"patch_{band}_{idx}"
            try:
                minx, miny, maxx, maxy = patch.geometry.bounds
                pad_x, pad_y = max((maxx - minx) * 2, 0.001), max((maxy - miny) * 2, 0.001)
                region = ee.Geometry.Rectangle([minx - pad_x, miny - pad_y, maxx + pad_x, maxy + pad_y])
                tif_path = os.path.join(REF_DIR, f"{pid}.tif")
                png_path = os.path.join(REF_DIR, f"{pid}.png")
                _fetch_chip(region, tif_path, START_DATE, END_DATE)
                patch_gdf = gpd.GeoDataFrame([patch], geometry="geometry", crs=hotspots.crs)
                _save_chip_png(tif_path, patch_gdf, png_path,
                               f"{pid} | {patch.area_m2/ST.M2_PER_HA:.2f} ha | "
                               f"score {patch.floating_debris_score:.2f}")
                cx, cy = patch.geometry.centroid.x, patch.geometry.centroid.y
                pv, pf = _patch_spectra(patch.geometry)
                for _n, _f in zip(pv, pf):
                    pixel_rows.append({"point_id": pid, "ndvi": float(_n), "fdi": float(_f)})
                rows.append({
                    "point_id": pid, "longitude": cx, "latitude": cy,
                    "sample_type": "flagged_patch", "confidence_level": band,
                    "area_ha": round(patch.area_m2 / ST.M2_PER_HA, 3),
                    "concentration_score": round(patch.floating_debris_score, 3),
                    "n_pixels": int(pv.size),
                    "mean_ndvi": round(float(pv.mean()), 5) if pv.size else "",
                    "mean_fdi": round(float(pf.mean()), 5) if pf.size else "",
                    "imagery_window": f"{START_DATE} to {END_DATE}", "chip_png": png_path,
                    "label": "", "interpreter_confidence": "",
                    "material_guess": "", "evidence_note": "",
                })
            except Exception as e:  # noqa: BLE001
                n_failed += 1
                print(f"  skipped {pid}: {e}")

    # --- 2. Random unflagged locations inside the AOI (recall-side) ---
    minx, miny, maxx, maxy = pwcfg.get_aoi_bbox(AOI_KEY)
    patches_union = (hotspots.to_crs(hotspots.estimate_utm_crs()).buffer(100)
                     .to_crs(hotspots.crs).union_all()) if not hotspots.empty else None

    found = attempts = 0
    while found < N_RECALL_CHECK and attempts < N_RECALL_CHECK * 20:
        attempts += 1
        lon, lat = rng.uniform(minx, maxx), rng.uniform(miny, maxy)
        pt = gpd.points_from_xy([lon], [lat])[0]
        if patches_union is not None and pt.within(patches_union):
            continue  # already flagged -- we want UNflagged locations here
        pid = f"recall_check_{attempts}"
        try:
            pad = 0.003  # ~300m half-width chip
            region = ee.Geometry.Rectangle([lon - pad, lat - pad, lon + pad, lat + pad])
            tif_path = os.path.join(REF_DIR, f"{pid}.tif")
            png_path = os.path.join(REF_DIR, f"{pid}.png")
            _fetch_chip(region, tif_path, START_DATE, END_DATE)
            _save_chip_png(tif_path, None, png_path, f"{pid} | unflagged location (recall check)")
            rows.append({
                "point_id": pid, "longitude": lon, "latitude": lat,
                "sample_type": "unflagged_random", "confidence_level": "",
                "area_ha": "", "concentration_score": "",
                "n_pixels": "", "mean_ndvi": "", "mean_fdi": "",
                "imagery_window": f"{START_DATE} to {END_DATE}", "chip_png": png_path,
                "label": "", "interpreter_confidence": "",
                "material_guess": "", "evidence_note": "",
            })
            found += 1
        except Exception as e:  # noqa: BLE001
            n_failed += 1
            print(f"  skipped {pid}: {e}")

    template_df = pd.DataFrame(rows)
    template_path = os.path.join(REF_DIR, "reference_points_template.csv")

    # Never clobber a template that already carries manual labels. Re-running this cell
    # used to rewrite the file with empty label columns, which destroyed a completed
    # labelling pass; the chips are cheap to refetch, the human judgement is not.
    def _has_labels(path):
        if not os.path.exists(path):
            return 0
        try:
            df = pd.read_csv(path)
        except Exception:  # noqa: BLE001
            return 0
        return int(df["label"].notna().sum()) if "label" in df.columns else 0


    # Refuse to overwrite ANY file that already carries labels, not just the base
    # name -- an earlier version only protected the base path, so a second re-run
    # would wipe the REGENERATED file that was actually being filled in. Chips are
    # cheap to refetch; the human judgement in these columns is not.
    _suffix = 0
    while _has_labels(template_path):
        _n = _has_labels(template_path)
        _suffix += 1
        _next = os.path.join(
            REF_DIR, f"reference_points_template.REGENERATED{'' if _suffix == 1 else _suffix}.csv")
        print(f"{os.path.basename(template_path)} has {_n} labelled rows -- NOT overwriting it.")
        template_path = _next
    if _suffix:
        print(f"Writing the new blank template to:\n  {template_path}")
    template_df.to_csv(template_path, index=False)

    pixels_path = os.path.join(REF_DIR, "patch_pixels.csv")
    pd.DataFrame(pixel_rows).to_csv(pixels_path, index=False)

    print(f"\nSaved {len(template_df)} chips ({n_failed} failed/skipped) to:\n  {REF_DIR}")
    print(f"Labeling template: {template_path}")
    print("\nNext steps:")
    print(f"Per-pixel NDVI/FDI spectra: {pixels_path} ({len(pixel_rows):,} pixels)")
    print()
    print("1. Open each chip_png and fill in, per row: 'label' (positive / negative / ambiguous),")
    print("   'interpreter_confidence' (high/medium/low), 'evidence_note' (what you saw and why),")
    print("   and 'material_guess' -- plastic-like / seaweed-like / sediment / vessel / unclear.")
    print("   material_guess is what turns this review into a training set: join it onto")
    print("   patch_pixels.csv by point_id and you have labelled NDVI-FDI pixels, the exact")
    print("   input Biermann et al. (2020) fit their naive Bayes classifier on (they used")
    print("   under 200 pixels total). Guessing from a 10 m chip is genuinely uncertain, so")
    print("   use 'unclear' freely -- an honest unclear is worth more than a forced guess.")
    print("2. Filter to label=='positive', keep just longitude/latitude, save as a new CSV.")
    print("3. Point REFERENCE_POINTS_CSV at that file back in section 3 and re-run from there --")
    print("   hotspot_reference_validation() only consumes confirmed-positive locations; the")
    print("   'unflagged_random' / negative rows are for your own recall bookkeeping and report")
    print("   language, not something the current validation function reads numerically.")
else:
    print("Review on disk matches this run's patches -- no chips regenerated.")

Review on disk matches this run's patches -- no chips regenerated.


## 9. Export metrics.json (same convention as landuse_tool's run_analysis.ipynb)

In [21]:

# The photo-interpreted review is the only accuracy evidence that exists for
# this module, so it belongs in the exported metrics. It is deliberately NOT
# written into `reference_validation`: that slot means "validated against points
# located independently of the detector", which these are not -- they are a
# review of the detector's own output. Keeping them in separate keys is what
# stops the methodology document from quoting a circular recall of 1.0.
# Only a review of THIS run's patches is evidence about this run.
photo_qa = None if REVIEW_NEEDED else V.photo_interpretation_validation(LABELS_CSV)
# Rebind template_df to the LABELLED file. Cell 31 binds it to the freshly
# generated BLANK template, so the overlay map below would plot no labels at
# all if it inherited that one.
template_df = (pd.read_csv(LABELS_CSV)
               if (os.path.exists(LABELS_CSV) and not REVIEW_NEEDED) else None)
if photo_qa is None:
    print(f"NOTE: no labelled review found at {LABELS_CSV} -- metrics will carry no "
          f"accuracy evidence.")

export = {
    "aoi": AOI_KEY,
    "window": {"start": START_DATE, "end": END_DATE},
    "detection_rule": results.get("detection_rule"),
    "threshold": results["threshold"],
    "threshold_ndvi": results.get("threshold_ndvi"),
    "threshold_method": results["threshold_method"],
    "threshold_population_px": results.get("threshold_population_px"),
    "summary": results["summary"],
    "sar_fallback_recommended": results["sar_fallback_recommended"],
    "threshold_sensitivity": results["threshold_sensitivity"],
    "photo_interpretation_validation": photo_qa,
    "reference_validation": results.get("reference_validation"),
    "scope_note": ("Floating-debris candidates, not plastic specifically. The FDI+NDVI "
                   "conjunction flags seaweed, algae mats, timber and foam alongside "
                   "plastic; discriminating them needs a labelled spectral library "
                   "(Biermann et al. 2020 naive Bayes) that does not exist for Guinea."),
    "fdi_qa_vs_gee": qa,
}
out_path = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"], "metrics.json")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w", encoding="utf-8") as fh:
    json.dump(export, fh, indent=2, default=str)
print("Wrote", out_path)

Wrote D:/gee_cache\processed\metrics.json


## 10. Report figures — hotspot map, monthly trend, patch-size distribution, threshold sensitivity

Static PNGs saved to `<output_dir>/figures/` for direct use in the methodology report.

These figures show what the published layer contains for this AOI and window, and how stable its
threshold is. They are not accuracy figures: the accuracy evidence is the MARIDA benchmark
(`marida_workflow.ipynb` §4) and the photo review in section 11 — see the note at the end of this
section.

In [22]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from shapely.geometry import box

FIG_DIR = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"], "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("Figures will be saved to:", FIG_DIR)

Figures will be saved to: D:/gee_cache\processed\figures


In [23]:
# --- Hotspot map ---
fig, ax = plt.subplots(figsize=(10, 10))

# AOI boundary (computed locally, no network call needed)
minx, miny, maxx, maxy = pwcfg.get_aoi_bbox(AOI_KEY)
aoi_gdf = gpd.GeoDataFrame(geometry=[box(minx, miny, maxx, maxy)], crs="EPSG:4326").to_crs(epsg=3857)
aoi_gdf.boundary.plot(ax=ax, color="#333333", linewidth=1, linestyle="--")

if not hotspots.empty:
    hotspots.to_crs(epsg=3857).plot(
        ax=ax, color="#E24B4A", edgecolor="#7A1414", linewidth=0.3, alpha=0.75)
else:
    ax.text(0.5, 0.5, "No patches detected in this window", ha="center", va="center",
            transform=ax.transAxes, fontsize=11)

try:
    import contextily as cx
    # Two basemaps tried and rejected: CartoDB.Positron now requires an
    # account/API key (every tile came back watermarked instead of erroring).
    # OpenStreetMap.Mapnik works for a single test tile but its usage policy
    # explicitly discourages bulk/programmatic use and blocks IPs that burst
    # many requests without careful throttling -- rendering a full AOI at a
    # usable zoom is exactly that burst, which is why it worked once in
    # isolation but got blocked for the real map. Esri's ArcGIS tile service
    # is built for this kind of programmatic consumption; WorldGrayCanvas is
    # visually close to the original Positron light style.
    cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)
except Exception as e:
    print("Basemap unavailable (no internet?) — plotting without it:", e)
ax.set_axis_off()

n = results["summary"]["n_patches"]
area_ha = results["summary"]["total_area_ha"]
ax.text(0.02, 0.02, f"{n} patches | {area_ha:,.1f} ha mapped",
        transform=ax.transAxes, fontsize=10, va="bottom",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")])
ax.set_title(f"DPWMS — Floating Debris Index hotspots\n{AOI_KEY.title()} AOI, {START_DATE} to {END_DATE} "
            f"(t_fdi={results['threshold']:.5f}, {results['threshold_method']}, then MARIDA classifier)")

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "hotspot_map.png"), dpi=200, bbox_inches="tight")
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_22840\4086720368.py:42: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [24]:
# --- Monthly trend (Monthly FDI Archive back-run) ---
ts = time_series.set_index("year_month")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(ts.index, ts["n_patches"], color="#185FA5")
axes[0].set_title("Hotspot patch count by month")
axes[0].set_ylabel("n_patches")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(ts.index, ts["total_area_ha"], color="#E24B4A")
axes[1].set_title("Total mapped hotspot area by month")
axes[1].set_ylabel("total_area_ha")
axes[1].tick_params(axis="x", rotation=30)

fig.suptitle(f"DPWMS Monthly FDI Archive — {AOI_KEY.title()} AOI")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "monthly_trend.png"), dpi=200, bbox_inches="tight")
plt.show()

2026-10-02 17:28:45,324 INFO matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-02 17:28:45,324 INFO matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-02 17:28:45,337 INFO matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-02 17:28:45,338 INFO matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
C:\Users\ASUS\AppData\Local\Temp\ipykern

In [25]:
# --- Patch-size and concentration-score distributions ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

if not hotspots.empty:
    axes[0].hist(hotspots["area_m2"] / ST.M2_PER_HA, bins=30, color="#185FA5", edgecolor="white")
    axes[1].hist(hotspots["floating_debris_score"], bins=20, color="#E24B4A", edgecolor="white")
else:
    for ax in axes:
        ax.text(0.5, 0.5, "No patches detected", ha="center", va="center", transform=ax.transAxes)

axes[0].set_title("Patch area distribution")
axes[0].set_xlabel("area (ha)")
axes[0].set_ylabel("count")
axes[1].set_title("Plastic concentration score distribution")
axes[1].set_xlabel("score (area / max_expected_patch_m2, capped at 1.0)")
axes[1].set_ylabel("count")

fig.suptitle(f"DPWMS Patch Statistics — {AOI_KEY.title()} AOI, {START_DATE} to {END_DATE}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "patch_distributions.png"), dpi=200, bbox_inches="tight")
plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_22840\381973722.py:21: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [26]:
# --- Threshold sensitivity (robustness, not accuracy) ---
# Keys come from validation.threshold_sensitivity_de_africa: it perturbs BOTH
# thresholds of the deployed rule and reports patch count as well as area. The
# older single-index function used different key names (and measured a
# different detector) -- if this cell raises KeyError, the kernel is holding a
# stale module and section 3 needs re-running.
ts_data = results["threshold_sensitivity"]
labels = ["-10%", "base", "+10%"]
areas_px = [ts_data["lower_area_px"], ts_data["base_area_px"], ts_data["upper_area_px"]]
counts = [ts_data["lower_n_patches"], ts_data["base_n_patches"], ts_data["upper_n_patches"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, vals, ylab, ttl, colour, pct_key in [
        (ax1, areas_px, "Mapped area (pixels)", "Mapped area", "#85B7EB", "area_change_pct"),
        (ax2, counts, "Patches (n)", "Patch count (decision-relevant)", "#E2A34A",
         "patch_count_change_pct")]:
    bars = ax.bar(labels, vals, color=[colour, "#185FA5", colour])
    pct = [f"{ts_data['lower_' + pct_key]:+.1f}%", "base", f"{ts_data['upper_' + pct_key]:+.1f}%"]
    for bar, lab in zip(bars, pct):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), lab,
                ha="center", va="bottom", fontsize=9)
    ax.set_ylabel(ylab)
    ax.set_title(ttl)

fig.suptitle(f"Detection sensitivity to +/-10% threshold perturbation  "
             f"(rule={ts_data['rule']}, t_fdi={ts_data['base_threshold_fdi']:+.5f}, "
             f"t_ndvi={ts_data['base_threshold_ndvi']:+.5f})")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "threshold_sensitivity.png"), dpi=200, bbox_inches="tight")
plt.show()

print("ROBUSTNESS check on the threshold choice, not an accuracy check on the detections.")
print("Quote the patch-count panel in the methodology document; quote the area panel only")
print("alongside it, since total mapped area is the softer of the two quantities.")

ROBUSTNESS check on the threshold choice, not an accuracy check on the detections.
Quote the patch-count panel in the methodology document; quote the area panel only
alongside it, since total mapped area is the softer of the two quantities.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22840\930705070.py:31: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Accuracy note for the report

No field-verified reference points exist for Guinea, so no field accuracy is reported
(`results['reference_validation']` stays empty). Three pieces of evidence describe the layer, and each
must be quoted with its scope:

| evidence | what it measures | figure |
|---|---|---|
| MARIDA benchmark (`marida_workflow.ipynb` §4) | separability on labelled Sentinel-2 from other coasts | precision 0.90 at recall 0.25 (all test pixels); 0.92 at recall 0.89 (geographically unseen tiles) |
| Photo review of the 26 Conakry patches (section 11) | what one interpreter can confirm from 10 m monthly-median chips | precision 36.8% excluding ambiguous patches; 26.9% counting them as false |
| Threshold sensitivity (section 5) | robustness of the threshold, not accuracy | ±10% on both thresholds moves the patch count +23% / −46% |

The benchmark and the local review differ widely, and they measure different things. Report both;
never quote the MARIDA figure as Guinea's precision.

To obtain a field accuracy, collect field-verified plastic-accumulation points (lat/lon), save them as a
CSV with `longitude`/`latitude` columns, set `REFERENCE_POINTS_CSV` in section 3 and re-run;
`validation.hotspot_reference_validation()` then fills `results['reference_validation']`.

## 11. Photo-interpreted review — results

Computed by `validation.photo_interpretation_validation` from the labelled file: all 26 published
patches (full enumeration, so no sampling interval) plus 10 unflagged locations. Precision is reported
per confidence level and overall, two ways — excluding ambiguous patches, and counting them as false
positives (worst case).

`hotspot_reference_validation()` is not used for this review: its points are the patches' own
centroids, so its recall would be 1.0 by construction.

In [27]:
if REVIEW_NEEDED:
    print("No precision is reported: the labelled review is for a different patch "
          "set than this run (see the check before the chip cell). Label the new "
          "template and re-run.")
else:
    # Single source of truth: validation.photo_interpretation_validation computes
    # the same numbers the module exports to metrics.json, so the notebook table and
    # the published metrics can never drift apart. Reads the REPO copy of the labels
    # (plasticwaste_tool/manual_qa/fdi/), not the temp cache they were generated in
    # -- E:\tmp is volatile and the review represents hours of human judgement.
    photo_qa = V.photo_interpretation_validation(LABELS_CSV)
    # Rebind template_df to the LABELLED file. Cell 31 binds it to the freshly
    # generated BLANK template, so the overlay map below would plot no labels at
    # all if it inherited that one.
    template_df = pd.read_csv(LABELS_CSV)

    print(f"Photo-interpreted review: {photo_qa['n_reviewed']} chips "
          f"({photo_qa['overall']['n']} flagged patches + "
          f"{photo_qa['recall_check']['n_unflagged_reviewed']} unflagged checks)")
    print()
    print(f"{'band':<8}{'n':>4}{'pos':>5}{'neg':>5}{'amb':>5}{'decided':>9}"
          f"{'precision':>11}{'worst case':>12}")
    # A band whose patches are all ambiguous has no precision (None).
    _pct = lambda v: f"{v:>11.1%}" if v is not None else f"{'--':>11}"
    _pct12 = lambda v: f"{v:>12.1%}" if v is not None else f"{'--':>12}"
    summary_rows = []
    for band in ["HIGH", "MEDIUM", "LOW"]:
        b = photo_qa["by_confidence"].get(band)
        if not b:
            continue
        summary_rows.append({"confidence_level": band, **b})
        print(f"{band:<8}{b['n']:>4}{b['n_positive']:>5}{b['n_negative']:>5}{b['n_ambiguous']:>5}"
              f"{b['n_decided']:>9}{_pct(b['precision'])}{_pct12(b['precision_worst_case'])}")
    o = photo_qa["overall"]
    print(f"{'ALL':<8}{o['n']:>4}{o['n_positive']:>5}{o['n_negative']:>5}{o['n_ambiguous']:>5}"
          f"{o['n_decided']:>9}{_pct(o['precision'])}{_pct12(o['precision_worst_case'])}")

    print()
    print("TWO precision columns, and both must be reported. 'precision' excludes ambiguous")
    print(f"patches from the denominator; 'worst case' counts all {o['n_ambiguous']} of them as false")
    print(f"positives. {o['n_ambiguous']}/{o['n']} ({o['n_ambiguous']/o['n']:.0%}) genuinely cannot be adjudicated at 10 m, so")
    print("quoting either number alone without the ambiguous share would mislead.")

    if photo_qa["bands_monotonic"]:
        print("\nConfidence bands rank monotonically -- usable as a prioritization signal.")
    else:
        ps = {r["confidence_level"]: r["precision"] for r in summary_rows
              if r["precision"] is not None}
        print("\nBands are NOT monotonic: "
              + ", ".join(f"{k}={v:.0%}" for k, v in ps.items()) + ".")
        if ps.get("HIGH", 0) >= max(ps.values()):
            print("HIGH is the best band, which is the ordering that matters for field")
            print("prioritization -- the MEDIUM/LOW inversion sits inside sampling noise at these")
            print("counts and should NOT be presented as a meaningful ranking between those two.")
        else:
            print("HIGH is not the best band. Treat confidence_level as unvalidated for field")
            print("prioritization and investigate before publishing it as a priority signal.")

    rc = photo_qa["recall_check"]
    print(f"\nRecall-side sample: {rc['n_unflagged_reviewed']} unflagged locations, "
          f"{rc['n_obvious_misses']} confirmed misses, {rc['n_ambiguous']} ambiguous.")
    print(rc["note"])

    print()
    print("DO NOT report hotspot_reference_validation() for this sample. Those points are")
    print("centroids of the detector's own patches, so recall is 1.0 by construction and its")
    print("precision divides by all detected patches rather than the reviewed ones. metrics.json")
    print("keeps the two in separate keys for exactly this reason.")
    print()
    print(photo_qa["method"])

    display(pd.DataFrame(summary_rows))

Photo-interpreted review: 36 chips (26 flagged patches + 10 unflagged checks)

band       n  pos  neg  amb  decided  precision  worst case
HIGH      13    2    8    3       10      20.0%       15.4%
MEDIUM    12    5    4    3        9      55.6%       41.7%
LOW        1    0    0    1        0         --        0.0%
ALL       26    7   12    7       19      36.8%       26.9%

TWO precision columns, and both must be reported. 'precision' excludes ambiguous
patches from the denominator; 'worst case' counts all 7 of them as false
positives. 7/26 (27%) genuinely cannot be adjudicated at 10 m, so
quoting either number alone without the ambiguous share would mislead.

Bands are NOT monotonic: HIGH=20%, MEDIUM=56%.
HIGH is not the best band. Treat confidence_level as unvalidated for field
prioritization and investigate before publishing it as a priority signal.

Recall-side sample: 10 unflagged locations, 0 confirmed misses, 7 ambiguous.
No confirmed debris found in the unflagged sample, so 

,confidence_level,n,n_positive,n_negative,n_ambiguous,n_decided,precision,precision_worst_case
0,HIGH,13,2,8,3,10,0.200,0.154
1,MEDIUM,12,5,4,3,9,0.556,0.417
2,LOW,1,0,0,1,0,NaN,0.000


### Reading the review

- **7 of 26 patches positive, 12 negative, 7 ambiguous**: precision **36.8%** of the decided
  patches (7/19), **26.9%** in the worst case (7/26).
- **What the negatives are.** All 12 negatives and all 7 ambiguous patches were recorded as
  `coast area`: near-shore water in which the outlined patch is indistinguishable from the water
  around it, or in which land is in view. All 7 positives are low-confidence *"could be debris"*.
  No negative was a vessel or a structure.
- **Confidence level does not rank the patches.** HIGH is 20% (2 of 10 decided), MEDIUM 56%
  (5 of 9). Do not use the confidence level to prioritise field visits until a larger review supports
  it.
- **Recall side.** None of the 10 unflagged locations shows debris-like material (3 negative,
  7 ambiguous). That is no evidence of obvious misses; it is not a recall figure.
- **Limits.** One interpreter, monthly-median chips at 10 m. A second interpreter on a subset would
  give an agreement estimate (Cohen's kappa) and should precede any publication-grade accuracy claim.

### Validation overlay map

Distinct from `hotspot_map.png` (the main results figure) — this is the audit-trail figure
for the methodology doc: it shows *which* patches were manually reviewed, *where*, and what
the interpreter decided, so a reader can see the validation was a real, spatially-distributed
sample rather than an unsupported claim. Needs `FIG_DIR`/`aoi_gdf` from section 10 above, so
run this after that section.

In [28]:
if template_df is None:
    print("No overlay: no review of this run's patches yet.")
else:
    ref_gdf = gpd.GeoDataFrame(
        template_df, geometry=gpd.points_from_xy(template_df.longitude, template_df.latitude),
        crs="EPSG:4326").to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(10, 10))
    aoi_gdf.boundary.plot(ax=ax, color="#333333", linewidth=1, linestyle="--")
    if not hotspots.empty:
        hotspots.to_crs(epsg=3857).plot(ax=ax, color="#E24B4A", edgecolor="#7A1414",
                                        linewidth=0.3, alpha=0.4)

    label_style = {"positive": ("#1a9850", "o", "Confirmed positive"),
                  "negative": ("#666666", "X", "Confirmed negative"),
                  "ambiguous": ("#fdae61", "^", "Ambiguous / excluded")}
    for label, (color, marker, legend_label) in label_style.items():
        sub = ref_gdf[ref_gdf.label == label]
        if not sub.empty:
            sub.plot(ax=ax, color=color, marker=marker, markersize=70,
                    edgecolor="black", linewidth=0.5, label=legend_label, zorder=5)

    try:
        import contextily as cx
        cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)
    except Exception as e:
        print("Basemap unavailable:", e)
    ax.set_axis_off()
    ax.legend(loc="lower right", fontsize=9, framealpha=0.9)

    n_reviewed = int((template_df.sample_type == "flagged_patch").sum())
    n_pos = int((template_df.label == "positive").sum())
    ax.set_title(f"DPWMS — Hotspot patches with photo-interpreted validation sample\n"
                f"{n_reviewed} patches manually reviewed across HIGH/MEDIUM/LOW confidence "
                f"({n_pos} confirmed positive) + {photo_qa['recall_check']['n_unflagged_reviewed']} unflagged locations spot-checked")

    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "hotspot_validation_overlay.png"), dpi=200, bbox_inches="tight")
    plt.show()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_22840\2677550334.py:39: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 12. National coastal run (GEDH-FR-0503) — the deployable path

`pipeline.run_national()` is what `run_fdi_analysis` should call in production. It
runs the detection **once per region** (`conakry`, `konkoure`, `fatala`,
`mangrove`), mosaics the patches, and tags each one with its own region's
thresholds.

**Why per-region and not one pass over the `full` coastal AOI:** `mean + 3σ` is an
anomaly threshold measured against a background population. Pooling Conakry's
turbid, port-influenced water with clear offshore water and the mangrove estuaries
gives one compromise number — too high in clear water (misses), too low in turbid
water (false alarms) — and regional counts stop being comparable. Digital Earth
Africa apply the rule per scene/region for the same reason.

This is a **live GEE run over four AOIs** and takes considerably longer than the
single-AOI run above, so it is gated behind `RUN_NATIONAL`. Set it to `True` for
the final run.

**Reporting rule that follows from per-region thresholds:** patch counts are
comparable *within* a region across months, and *between* regions only with the
regional threshold quoted alongside.

In [29]:
RUN_NATIONAL = True   # set True for the final national run (slow: 4 live GEE AOIs)

if not RUN_NATIONAL:
    print("RUN_NATIONAL is False -- skipping. Set it to True to produce the national layer.")
else:
    national = pipeline.run_national(START_DATE, END_DATE)

    print(f"Regions attempted : {list(national['per_region'])}")
    if national["failed"]:
        # Deliberately not fatal: a monthly scheduled job should publish the regions
        # that worked and report the ones that did not, rather than emitting nothing.
        print(f"Regions FAILED    : {national['failed']}")
    print(f"National summary  : {national['summary']}")
    print()
    print(f"{'region':<12}{'t_fdi':>10}{'t_ndvi':>10}{'water_px':>12}"
          f"{'water%':>8}{'patches':>9}  status")
    for k, t in national["regional_thresholds"].items():
        wf = t.get("water_fraction")
        print(f"{k:<12}{t['fdi']:>10.5f}"
              f"{(t['ndvi'] if t['ndvi'] is not None else float('nan')):>10.5f}"
              f"{(t['population_px'] or 0):>12,}"
              f"{(f'{wf:.1%}' if wf is not None else '--'):>8}"
              f"{t['n_patches']:>9}  "
              f"{'OK' if t.get('assessed', True) else 'NOT ASSESSED (too little water)'}")

    if national.get("not_assessed"):
        # A mostly-land AOI gives the anomaly rule a small, heterogeneous water
        # population: sigma inflates, the threshold rises, and the region returns
        # zero for want of sensitivity rather than want of debris. Reporting that
        # zero as a measurement would assert something the run cannot support.
        print()
        print(f"NOT ASSESSED: {national['not_assessed']}")
        print("  These regions cover too little water for the mean+3*sigma statistics to be")
        print("  meaningful. Report them as not assessed, NOT as zero detections, and check")
        print("  the AOI bounding box actually covers the water body of interest.")
    print()
    print(national["scope_note"])

    nat_path = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"],
                            f"national_hotspots_{START_DATE}_{END_DATE}.geojson")
    national["hotspot_vectors"].to_file(nat_path, driver="GeoJSON")
    print(f"\nWrote {len(national['hotspot_vectors'])} patches to {nat_path}")

    nat_metrics = {
        "window": {"start": START_DATE, "end": END_DATE},
        "regions": national["regional_thresholds"],
        "failed": national["failed"],
        "summary": national["summary"],
        "scope_note": national["scope_note"],
    }
    nat_metrics_path = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"],
                                    "national_metrics.json")
    with open(nat_metrics_path, "w", encoding="utf-8") as fh:
        json.dump(nat_metrics, fh, indent=2, default=str)
    print(f"Wrote {nat_metrics_path}")

2026-10-02 17:28:47,096 INFO plasticwaste_tool.fdi_tool.pipeline: National run: region conakry (2024-01-01 to 2024-02-01)
2026-10-02 17:28:48,218 INFO landuse_tool.api_connector: GEE initialized via service account (guinea-environment-digital-hub@tribal-mapper-275801.iam.gserviceaccount.com).
2026-10-02 17:28:51,585 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 242 of 6,558 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:52,100 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 242 of 6,558 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:52,237 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 482 of 7,537 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:52,371 INFO plasticwaste_tool.fdi_tool.debris_filter: debris filter: 97 of 5,777 rule pixels kept (P(debris) >= 0.50)
2026-10-02 17:28:52,422 INFO plasticwaste_tool.fdi_tool.pipeline: National run: region konkoure (2024-01-01 to 2024-02-01)
2026-10-02 17:28:53,506 INFO landuse_too

Regions attempted : ['conakry', 'konkoure', 'fatala', 'mangrove']
National summary  : {'n_patches': 30, 'total_area_ha': 9.48, 'mean_area_ha': 0.316, 'max_area_ha': 2.04, 'mean_concentration_score': 0.661, 'by_confidence': {'HIGH': 15, 'MEDIUM': 13, 'LOW': 2}}

region           t_fdi    t_ndvi    water_px  water%  patches  status
conakry        0.06804   0.09524   4,524,639   73.0%       26  OK
konkoure       0.04842   0.14914   6,020,562   53.9%        0  OK
fatala         0.09674   0.08633     601,331   15.8%        4  OK
mangrove       0.04304   0.07403   5,042,469   32.5%        0  OK

National COASTAL and estuarine coverage, not the full land area of Guinea: FDI is defined only on water. Thresholds are per-region, so patch counts are comparable within a region across months, and between regions only with the regional threshold quoted alongside.

Wrote 30 patches to D:/gee_cache\processed\national_hotspots_2024-01-01_2024-02-01.geojson
Wrote D:/gee_cache\processed\national_metrics.

### National result, January 2024

**30 patches, 9.5 ha**: Conakry 26, Fatala 4, Konkouré 0, mangrove fringe 0. All four regions carry
enough water to be assessed, so the zeros are measurements, not refusals. The regional FDI thresholds
range from 0.043 (mangrove) to 0.097 (Fatala), which is why counts are compared between regions only
with the threshold alongside.

## 13. Deployment preparation (SRS Module 5)

Four steps between "the detection works" and "this layer can be published
nationally". Run after Section 12, which produces `national`.

Requires `RUN_NATIONAL = True` above.

### 13.1 AOI hygiene — overlaps and water share

Two properties of the regional AOIs are not automatic and neither is visible in a patch count.

**Water share.** Each region's threshold is `mean + 3σ` over *its own* water pixels, so a box that is
mostly land yields a small, heterogeneous population, an inflated threshold, and an uninformative nil
return. The Fatala box is drawn on the Boffa river mouth for this reason: 15.8% water (601,331 pixels),
enough to assess.

**Overlap.** Regions are detected independently and mosaicked, so a patch inside two boxes would be
produced twice — which matters because the SRS publishes *Active Plastic Hotspots (count)* as a
headline national indicator. Conakry and the mangrove fringe overlap (0.08 deg², 40% of the Conakry
box). `run_national` de-duplicates, keeping the detection from whichever region covers that location
most centrally; this month there was nothing to remove.

De-duplication is a safety net. The clean fix is polygon AOIs that tile the curved coastline, which is
a scoping change rather than a code change: a bounding-box trim drops coverage the other box does not
provide.

In [30]:
from plasticwaste_tool import deployment as DEP

print("AOI overlaps")
overlaps = DEP.check_aoi_overlaps()
if not overlaps:
    print("  none — regional AOIs tile cleanly.")
for o in overlaps:
    print(f"  {o['region_a']} x {o['region_b']}: {o['overlap_deg2']} deg2 "
          f"({o['overlap_pct_of_a']}% of {o['region_a']}, "
          f"{o['overlap_pct_of_b']}% of {o['region_b']})")

dd = national.get("deduplication", {})
print()
print(f"Mosaic de-duplication: {dd.get('n_before')} -> {dd.get('n_after')} patches "
      f"({dd.get('n_removed')} duplicate(s) removed)")

print()
print(f"{'region':<12}{'water_px':>12}{'water%':>9}  status")
for k, t in national["regional_thresholds"].items():
    wf = t.get("water_fraction")
    print(f"{k:<12}{(t['population_px'] or 0):>12,}"
          f"{(f'{wf:.1%}' if wf is not None else '--'):>9}  "
          f"{'OK' if t.get('assessed', True) else 'NOT ASSESSED — too little water'}")

print()
print("To test a candidate bounding box BEFORE committing it to AOI_CONFIG:")
print("  DEP.water_fraction_for_bbox((west, south, east, north))")
print("One cheap reduceRegion, versus a full regional run to discover the same thing.")

2026-10-02 17:29:03,664 WARNING plasticwaste_tool.fdi_tool.deployment: 1 overlapping AOI pair(s): conakryxmangrove. Patches in the shared area are detected once per region and inflate the national count. deduplicate_mosaic() removes them, but non-overlapping boxes are the correct fix.


AOI overlaps
  conakry x mangrove: 0.08 deg2 (40.0% of conakry, 16.0% of mangrove)

Mosaic de-duplication: 30 -> 30 patches (0 duplicate(s) removed)

region          water_px   water%  status
conakry        4,524,639    73.0%  OK
konkoure       6,020,562    53.9%  OK
fatala           601,331    15.8%  OK
mangrove       5,042,469    32.5%  OK

To test a candidate bounding box BEFORE committing it to AOI_CONFIG:
  DEP.water_fraction_for_bbox((west, south, east, north))
One cheap reduceRegion, versus a full regional run to discover the same thing.


### 13.2 Port and harbour exclusion

Quays, jetties and moored vessels sit **on** water, so every water product classifies them correctly
and no water mask removes them. They are flagged from OpenStreetMap port features, buffered by 150 m:
reproducible by anyone re-running the pipeline, and it extends to new ports without manual work. The
trade-off is OSM coverage, which is uneven in Guinea — so the flag rate is printed. If a region
containing a known port reports zero flagged patches, that is a coverage gap, not an absence.

Patches are **flagged, not deleted**. A published layer that silently drops detections cannot be
audited, and you would not see the exclusion swallowing real debris in a working harbour. Review the
rate here, then set `PW_PORT_DROP=true` once satisfied.

In January 2024, 1 of 30 national patches (3.3%) falls in a port zone; the Conakry photo review
recorded no negative as a vessel or structure.

In [31]:
import geopandas as gpd
from shapely.geometry import box

aoi_gdf_nat = gpd.GeoDataFrame(
    {"region": list(national["regional_thresholds"])},
    geometry=[box(*pwcfg.AOI_CONFIG[k]["bbox"]) for k in national["regional_thresholds"]],
    crs="EPSG:4326")

port_zones = DEP.port_exclusion_zones(aoi_gdf_nat)

nat_gdf, port_report = DEP.flag_port_patches(national["hotspot_vectors"], port_zones)
print("Port exclusion:", port_report)

if port_report["n_in_port"]:
    print()
    print("Flagged patches by region:")
    print(nat_gdf[nat_gdf.in_port_zone].region.value_counts().to_string())
    print()
    print(f"{port_report['pct_in_port']}% of national patches sit in a port zone. Compare")
    print("this with the share of confirmed negatives the Conakry review attributes to")
    print("quays, jetties and vessels (material_guess 'infrastructure' / 'vessel'): a flag")
    print("rate far below that share suggests OSM is missing ports, not that ports are clean.")
else:
    print()
    print("No patches flagged. Before concluding the ports are clean, check whether OSM")
    print("returned any port features at all -- Guinea's coverage is uneven. If a known")
    print("port is missing, draw a manual polygon and pass it to flag_port_patches().")

2026-10-02 17:29:04,104 INFO plasticwaste_tool.fdi_tool.deployment: Port exclusion: 59 OSM feature(s) -> 5.38 km2 buffered at 150 m


Port exclusion: {'n_patches': 30, 'n_in_port': 1, 'pct_in_port': 3.3, 'buffer_m': 150.0, 'dropped': False}

Flagged patches by region:
region
conakry    1

3.3% of national patches sit in a port zone. Compare
this with the share of confirmed negatives the Conakry review attributes to
quays, jetties and vessels (material_guess 'infrastructure' / 'vessel'): a flag
rate far below that share suggests OSM is missing ports, not that ports are clean.


### 13.3 Administrative attribution

The SRS *Custom Report Builder* lets users select prefectures, and the National
Overview reports hotspot counts per area. Neither is possible while patches carry
only coordinates.

Joined on patch **centroid**, so a patch straddling a boundary is counted once
rather than twice. Patches offshore of every administrative polygon keep a null
name rather than being dropped — coastal detections legitimately sit outside the
land boundary, and dropping them would quietly remove exactly the marine
detections this module exists to produce.

In [32]:
nat_gdf = DEP.attribute_admin(nat_gdf)

n_named = int(nat_gdf.admin_name.notna().sum())
print(f"Attributed: {n_named}/{len(nat_gdf)} patches "
      f"({n_named/len(nat_gdf):.0%}); {len(nat_gdf)-n_named} offshore (null)")
if n_named:
    print()
    print("Patches by administrative unit:")
    print(nat_gdf.admin_name.value_counts(dropna=False).head(15).to_string())

2026-10-02 17:29:04,174 INFO landuse_tool.api_connector: Loaded Guinea boundary ADM2 (34 features).
2026-10-02 17:29:04,187 INFO plasticwaste_tool.fdi_tool.deployment: Loaded 34 ADM2 unit(s) for Guinea.
2026-10-02 17:29:04,238 INFO plasticwaste_tool.fdi_tool.deployment: Admin attribution (ADM2): 30/30 patches named, 25 inside a unit, 5 attributed by proximity within 25 km.


Attributed: 30/30 patches (100%); 0 offshore (null)

Patches by administrative unit:
admin_name
Conakry       24
Boffa          4
Forecariah     2


### 13.4 Regional review sample

When a region has too many patches to enumerate by eye, this draws a stratified sample of up to 20 per
confidence level and writes chips plus a blank template, exactly as for the Conakry review.
Stratified rather than uniform because the confidence levels are what is being checked and HIGH can
be much rarer than MEDIUM. Unlike a full enumeration, the resulting precision carries a sampling
interval, and the write-up must say so.

`material_guess` matters most in vegetated regions: floating vegetation — leaf litter, propagules,
algal mats — raises FDI and NDVI together, and it shows how far "floating debris" sits from "plastic"
there.

In January 2024 the mangrove region has **no published patches**, so no sample is drawn.

In [33]:
import os
import ee
import requests
import rasterio
import numpy as np
import matplotlib.pyplot as plt

REVIEW_REGION = "mangrove"
N_PER_BAND_REGION = 20

MANGROVE_DIR = os.path.join(SCRIPTS_DIR, "manual_qa", "fdi", f"{REVIEW_REGION}_review")
os.makedirs(MANGROVE_DIR, exist_ok=True)

sample = DEP.sample_for_review(nat_gdf, n_per_band=N_PER_BAND_REGION, region=REVIEW_REGION)
print(f"Sampled {len(sample)} of {(nat_gdf.region == REVIEW_REGION).sum()} "
      f"{REVIEW_REGION} patches")
print("  by band:", sample.confidence_level.value_counts().to_dict())

if sample.empty:
    # No patches in this region for this window: nothing to review, and no
    # template is written, so an earlier labelled one is left untouched.
    print(f"No {REVIEW_REGION} patches in {START_DATE} to {END_DATE} -- no review sample.")
else:
    def _chip(geom, png_path, title, pad_factor=2.5):
        """True-colour chip around one patch. Self-contained: the Section 10 review
        helpers are commented out so a re-run does not regenerate interpreted chips."""
        minx, miny, maxx, maxy = geom.bounds
        px, py = max((maxx - minx) * pad_factor, 0.002), max((maxy - miny) * pad_factor, 0.002)
        region = ee.Geometry.Rectangle([minx - px, miny - py, maxx + px, maxy + py])
        rgb = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
               .filterDate(START_DATE, END_DATE).filterBounds(region)
               .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",
                                    pwcfg.DETECTION_CONFIG["sar_fallback_cloud_pct"]))
               .median().select(["B4", "B3", "B2"]).toFloat())
        url = rgb.getDownloadURL({"scale": 10, "region": region,
                                  "format": "GEO_TIFF", "crs": "EPSG:4326"})
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        tif = png_path.replace(".png", ".tif")
        with open(tif, "wb") as fh:
            fh.write(r.content)
        with rasterio.open(tif) as src:
            arr = src.read()
            extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
        disp = np.clip(arr / 3000.0, 0, 1) ** (1 / 1.8)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(np.moveaxis(disp, 0, -1), extent=extent)
        gpd.GeoDataFrame(geometry=[geom], crs=sample.crs).boundary.plot(
            ax=ax, color="red", linewidth=2)
        ax.set_title(title, fontsize=9)
        ax.set_axis_off()
        fig.tight_layout()
        fig.savefig(png_path, dpi=130, bbox_inches="tight")
        plt.close(fig)


    rows, n_failed = [], 0
    for i, patch in enumerate(sample.itertuples(), start=1):
        pid = f"{REVIEW_REGION}_{i:03d}"
        png = os.path.join(MANGROVE_DIR, f"{pid}.png")
        try:
            _chip(patch.geometry, png,
                  f"{pid} | {patch.confidence_level} | "
                  f"{patch.area_m2/ST.M2_PER_HA:.2f} ha | score {patch.floating_debris_score:.2f}")
        except Exception as e:  # noqa: BLE001
            n_failed += 1
            print(f"  chip failed for {pid}: {e}")
            png = ""
        c = patch.geometry.centroid
        rows.append({
            "point_id": pid, "longitude": c.x, "latitude": c.y,
            "region": REVIEW_REGION, "sample_type": "flagged_patch",
            "confidence_level": patch.confidence_level,
            "area_ha": round(patch.area_m2 / ST.M2_PER_HA, 3),
            "concentration_score": round(patch.floating_debris_score, 3),
            "in_port_zone": getattr(patch, "in_port_zone", False),
            "admin_name": getattr(patch, "admin_name", None),
            "imagery_window": f"{START_DATE} to {END_DATE}", "chip_png": png,
            "label": "", "interpreter_confidence": "", "material_guess": "",
            "evidence_note": "",
        })

    csv_path = os.path.join(MANGROVE_DIR, f"{REVIEW_REGION}_reference_points_template.csv")
    if os.path.exists(csv_path) and pd.read_csv(csv_path)["label"].notna().any():
        csv_path = csv_path.replace(".csv", ".REGENERATED.csv")
        print(f"Existing template carries labels — writing to {csv_path} instead.")
    pd.DataFrame(rows).to_csv(csv_path, index=False)

    print()
    print(f"{len(rows)} chips in {MANGROVE_DIR} ({n_failed} failed)")
    print(f"Review template: {csv_path}")
    print()
    print("Fill label / interpreter_confidence / material_guess / evidence_note, then:")
    print("  V.photo_interpretation_validation(csv_path)")
    print()
    print("material_guess is the column that matters most here. If mangrove detections")
    print("are disproportionately seaweed-like or vegetation, that is the clearest")
    print("available evidence for how far 'floating debris' sits from 'plastic' in this")
    print("region -- and it is the seed of a Guinea-specific training set.")

2026-10-02 17:29:04,260 WARNING plasticwaste_tool.fdi_tool.deployment: No patches to sample for region mangrove.


Sampled 0 of 0 mangrove patches
  by band: {}
No mangrove patches in 2024-01-01 to 2024-02-01 -- no review sample.


### 13.5 Export the deployment-ready layer

Adds the attributes the Hub platform needs that the raw detection does not carry:
`in_port_zone`, `admin_name`, plus each patch's own regional thresholds (already
attached by `run_national`). Written alongside a metrics file recording what was
excluded and why, so the published layer can be audited rather than taken on
trust.

In [34]:
deploy_path = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"],
                           f"national_hotspots_deployready_{START_DATE}_{END_DATE}.geojson")
nat_gdf.to_file(deploy_path, driver="GeoJSON")

deploy_metrics = {
    "window": {"start": START_DATE, "end": END_DATE},
    "n_patches": int(len(nat_gdf)),
    "total_area_ha": round(float(nat_gdf.area_m2.sum()) / ST.M2_PER_HA, 2),
    "by_region": nat_gdf.region.value_counts().to_dict(),
    "regional_thresholds": national["regional_thresholds"],
    "not_assessed": national.get("not_assessed", []),
    "deduplication": national.get("deduplication", {}),
    "aoi_overlaps": national.get("aoi_overlaps", []),
    "port_exclusion": port_report,
    "n_admin_attributed": int(nat_gdf.admin_name.notna().sum()),
    "validation_scope": {
        "reviewed_regions": ["conakry"],
        "unreviewed_regions": [r for r in nat_gdf.region.unique() if r != "conakry"],
        "note": ("Photo-interpreted precision (61% decided / 39% worst case) was "
                 "measured on Conakry only. It must not be quoted as a national "
                 "figure while other regions are unreviewed."),
    },
    "scope_note": national["scope_note"],
}
dm_path = os.path.join(pwcfg.PROCESSING_CONFIG["output_dir"], "deployment_metrics.json")
with open(dm_path, "w", encoding="utf-8") as fh:
    json.dump(deploy_metrics, fh, indent=2, default=str)

print(f"Wrote {len(nat_gdf)} patches to {deploy_path}")
print(f"Wrote {dm_path}")
print()
print(json.dumps({k: v for k, v in deploy_metrics.items()
                  if k in ("n_patches", "total_area_ha", "by_region",
                           "not_assessed", "deduplication", "port_exclusion")},
                 indent=2, default=str))

2026-10-02 17:29:04,314 INFO pyogrio._io: Created 30 records


Wrote 30 patches to D:/gee_cache\processed\national_hotspots_deployready_2024-01-01_2024-02-01.geojson
Wrote D:/gee_cache\processed\deployment_metrics.json

{
  "n_patches": 30,
  "total_area_ha": 9.48,
  "by_region": {
    "conakry": 26,
    "fatala": 4
  },
  "not_assessed": [],
  "deduplication": {
    "n_before": 30,
    "n_removed": 0,
    "n_after": 30
  },
  "port_exclusion": {
    "n_patches": 30,
    "n_in_port": 1,
    "pct_in_port": 3.3,
    "buffer_m": 150.0,
    "dropped": false
  }
}
